<a href="https://colab.research.google.com/github/KatherinePalaciosaCortez/CLAVE_I_Parcial4/blob/main/notebooks/Clustering_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Cargar datos
df = pd.read_csv("https://raw.githubusercontent.com/KatherinePalaciosaCortez/CLAVE_I_Parcial4/refs/heads/main/data/raw/clave_I_agrupacion.csv")


In [3]:

# 2. Validación básica
print(df.shape)
print(df.info())
print(df.isnull().sum())
print("Duplicados:", df.duplicated().sum())
print(df.describe())



(262, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262 entries, 0 to 261
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   registro_id       262 non-null    object 
 1   edad              262 non-null    int64  
 2   ingresos          262 non-null    int64  
 3   frecuencia_uso    262 non-null    float64
 4   gasto_promedio    262 non-null    float64
 5   satisfaccion      261 non-null    float64
 6   reclamos          262 non-null    int64  
 7   antiguedad_meses  262 non-null    int64  
dtypes: float64(3), int64(4), object(1)
memory usage: 16.5+ KB
None
registro_id         0
edad                0
ingresos            0
frecuencia_uso      0
gasto_promedio      0
satisfaccion        1
reclamos            0
antiguedad_meses    0
dtype: int64
Duplicados: 0
             edad     ingresos  frecuencia_uso  gasto_promedio  satisfaccion  \
count  262.000000   262.000000      262.000000      262.000000    

In [5]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Cargar datos
df = pd.read_csv("https://raw.githubusercontent.com/KatherinePalaciosaCortez/CLAVE_I_Parcial4/refs/heads/main/data/raw/clave_I_agrupacion.csv")

# 3. Valores atípicos por media +/- 2 desviaciones estándar
for columna in ["gasto_promedio", "ingresos"]:
    media = df[columna].mean()
    desviacion = df[columna].std()
    limite_inferior = media - 2 * desviacion
    limite_superior = media + 2 * desviacion
    atipicos = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]
    print(columna, limite_inferior, limite_superior, len(atipicos))


gasto_promedio -16.45610910545122 191.76519307491685 6
ingresos 266.30251050611264 2045.2394742267118 7


In [8]:
# 4. Variables del modelo
X = df[["ingresos", "gasto_promedio", "edad"]]


In [9]:
# 5. K-Means con K=2
modelo_k2 = KMeans(n_clusters=2, random_state=0, n_init=10)
df["cluster_k2"] = modelo_k2.fit_predict(X)
print(pd.DataFrame(modelo_k2.cluster_centers_, columns=X.columns))
print("Silhouette K=2:", silhouette_score(X, df["cluster_k2"]))



      ingresos  gasto_promedio       edad
0  1808.865672      156.505224  52.567164
1   931.374359       63.998154  38.158974
Silhouette K=2: 0.6829479029339323


In [10]:
# 6. K-Means con K=4
modelo_k4 = KMeans(n_clusters=4, random_state=0, n_init=10)
df["cluster_k4"] = modelo_k4.fit_predict(X)
centroides_k4 = pd.DataFrame(modelo_k4.cluster_centers_, columns=X.columns)
print(centroides_k4)
print(df["cluster_k4"].value_counts())
print("Silhouette K=4:", silhouette_score(X, df["cluster_k4"]))



      ingresos  gasto_promedio       edad
0  1859.568966      162.168276  53.172414
1   915.551020       63.360816  38.877551
2  1255.672131       94.354098  42.196721
3   636.377778       35.439333  33.222222
cluster_k4
1    98
2    61
0    58
3    45
Name: count, dtype: int64
Silhouette K=4: 0.5306790737789383


In [15]:
# 7. Identificar clúster de mayor riesgo
centroides_k4["riesgo"] = centroides_k4["ingresos"] + centroides_k4["gasto_promedio"]
cluster_riesgo = centroides_k4["riesgo"].idxmax()
print("Clúster de mayor riesgo:", cluster_riesgo)


Clúster de mayor riesgo: 0


In [14]:
# 8. Filtrar personas del clúster de riesgo
personas_riesgo = df[df["cluster_k4"] == cluster_riesgo]
print(personas_riesgo[["ingresos", "gasto_promedio", "edad"]].describe())
personas_riesgo.to_csv("personas_riesgo_cluster_kmeans.csv", index=False)


          ingresos  gasto_promedio       edad
count    58.000000       58.000000  58.000000
mean   1859.568966      162.168276  53.172414
std     196.264422       18.985070   6.184695
min    1598.000000      124.810000  38.000000
25%    1754.000000      150.052500  48.250000
50%    1834.000000      161.800000  54.000000
75%    1903.750000      175.322500  57.000000
max    2900.000000      207.310000  66.000000
